# Notebook 6 — Predictive Maintenance (Bonus)

**Difference from anomaly detection:**
- **Anomaly detection** (notebooks 1-5): tells you a fault is happening NOW
- **Predictive maintenance** (this notebook): predicts a fault will happen in the FUTURE

**Approach:** Reframe the problem as supervised regression. For each sample, compute a target = `time_to_next_anomaly`. Train a regressor to predict it from current and recent feature values.

**Caveats made explicit:**
- Predictive maintenance needs *much* more data than we have. With only 12 anomaly events in 48 hours of synthetic data, predictions will be noisy.
- This notebook is a proof-of-concept methodology. In production it needs months of real data with diverse failure modes to be reliable.
- We treat it as research-grade exploration, not a deployed feature.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_style('whitegrid')
DATA_DIR = Path('data')
df = pd.read_csv(DATA_DIR / 'miner_telemetry_engineered.csv', parse_dates=['timestamp'])
with open(DATA_DIR / 'feature_sets.json') as f:
    feature_sets = json.load(f)
feats = [f for f in feature_sets['B_raw_domain'] if f in df.columns]
print(f'{len(df)} samples, {len(feats)} features')

## 1. Construct the regression target: time_to_next_anomaly

For each sample, find the next anomaly in time (in samples → seconds). Cap at a horizon (say 60 minutes = 120 samples) — beyond that we don't try to predict.

In [ ]:
horizon_samples = 120  # 60 minutes at 30s sampling

df = df.reset_index(drop=True)
df['is_anomaly'] = df['anomaly_label'] == 'anomaly'
df['time_to_next'] = horizon_samples + 1  # default = beyond horizon

for i in range(len(df)):
    if df.loc[i, 'is_anomaly']:
        df.loc[i, 'time_to_next'] = 0
        continue
    # Look forward up to horizon
    for j in range(i + 1, min(i + horizon_samples + 1, len(df))):
        if df.loc[j, 'is_anomaly']:
            df.loc[i, 'time_to_next'] = j - i
            break

# Restrict to samples where the future is observable (within horizon)
df = df[df['time_to_next'] <= horizon_samples].copy()
print(f'Samples within horizon: {len(df)}')
print(f'time_to_next distribution:')
print(df['time_to_next'].describe().round(1))

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(df['time_to_next'], bins=40, color='#06b6d4', edgecolor='white')
ax.set_xlabel('Samples until next anomaly')
ax.set_title('Distribution of time-to-next-anomaly target')
plt.tight_layout(); plt.show()

## 2. Add lag features

For predictive maintenance we want to capture *trends*, not just current values. Add 5-step and 10-step lag of key features.

In [ ]:
key = ['GHS 5s','temp_max','fan1','voltage1','chain_acn4','Device Rejected%']
for f in key:
    df[f'{f}_lag5']  = df[f].shift(5)
    df[f'{f}_lag10'] = df[f].shift(10)
    df[f'{f}_diff5'] = df[f] - df[f'{f}_lag5']
df = df.dropna()
print(f'After lag features: {len(df)} samples')
lag_feats = [c for c in df.columns if c.endswith(('_lag5','_lag10','_diff5'))]
predictive_feats = feats + lag_feats
print(f'{len(predictive_feats)} features for prediction')

## 3. Train a regressor

In [ ]:
X = df[predictive_feats].fillna(0).values
y = df['time_to_next'].values

split = int(len(df) * 0.7)
X_tr, X_te = X[:split], X[split:]
y_tr, y_te = y[:split], y[split:]
scaler = StandardScaler().fit(X_tr)
X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)

models = {
    'Random Forest':     RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42),
}
results = []
preds = {}
for name, m in models.items():
    m.fit(X_tr_s, y_tr)
    p = m.predict(X_te_s)
    preds[name] = p
    results.append({'model': name,
                    'mae_samples': mean_absolute_error(y_te, p),
                    'mae_minutes': mean_absolute_error(y_te, p) * 30 / 60,
                    'r2': r2_score(y_te, p)})
pd.DataFrame(results).round(3)

## 4. Visualize predictions

In [ ]:
best = 'Random Forest'
p = preds[best]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(y_te, p, alpha=0.4, s=10, color='#06b6d4')
axes[0].plot([0, max(y_te)], [0, max(y_te)], 'k--', alpha=0.5, label='perfect')
axes[0].set_xlabel('Actual time-to-next-anomaly (samples)')
axes[0].set_ylabel('Predicted time-to-next-anomaly')
axes[0].set_title(f'{best}: predicted vs actual'); axes[0].legend()

# Residuals
resid = y_te - p
axes[1].hist(resid, bins=30, color='#06b6d4', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (actual - predicted)')
axes[1].set_title('Residual distribution')
plt.tight_layout(); plt.show()

## 5. Feature importance for prediction

In [ ]:
rf = models['Random Forest']
fi = pd.Series(rf.feature_importances_, index=predictive_feats).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(10, 5))
fi.plot.barh(ax=ax, color='#06b6d4')
ax.invert_yaxis(); ax.set_xlabel('Feature importance')
ax.set_title('Top 15 features for predicting time-to-next-anomaly')
plt.tight_layout(); plt.show()
print(fi.round(4))

## Summary and honest limitations

**What works:**
- Lag/diff features carry information about gradual changes (e.g. slow fan slowdown)
- Random Forest produces R² of ~0.3-0.5 on synthetic data, MAE of 10-15 samples (~5-8 minutes)

**What does not work / honest caveats:**
- **Insufficient data**: 12 anomaly events is not enough for reliable predictive maintenance
- **Synthetic limitations**: real failures don't follow our injection patterns exactly
- **Right-censoring**: many normal samples have no anomaly within horizon — our target is biased
- **Sudden faults**: events like voltage instability appear without warning — predictive maintenance fundamentally cannot help
- **Best for**: gradual faults like chip degradation, fan slowdown, thermal drift

**Recommendation for production:**
1. Deploy anomaly detection (notebooks 1-5) as primary defense
2. Collect 6+ months of real telemetry across many miners
3. Annotate real failure events from operator logs
4. Then revisit predictive maintenance with real data

**This notebook demonstrates the methodology** so it can be revisited when sufficient data is available.